# Baseline C — Cross-lingual aligned (MUSE)

## Setup

In [1]:
# Ensure the repo root (parent of notebooks/) is on sys.path so that
# `import baselines` resolves correctly regardless of how the kernel was
# launched or what the CWD is.
import sys
import pathlib

_NOTEBOOK_DIR = pathlib.Path(__file__).resolve().parent if '__file__' in dir() else pathlib.Path('.').resolve()
_REPO_ROOT = _NOTEBOOK_DIR.parent if _NOTEBOOK_DIR.name == 'notebooks' else _NOTEBOOK_DIR

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f'Repo root:   {_REPO_ROOT}')
print(f'sys.path[0]: {sys.path[0]}')

Repo root:   /home/anna/ph-project-ssa.6
sys.path[0]: /home/anna/ph-project-ssa.6


In [2]:
import warnings

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.cluster.hierarchy import dendrogram
from sklearn.metrics.pairwise import cosine_distances

from baselines import SUPPORTED_LANGUAGES
from baselines.vectors import load_fasttext
from baselines.distances import extract_term_vectors, cosine_distance_matrix
from baselines.clustering import agglomerative_linkage
from baselines.topology import rips_barcode, barcode_features

# Suppress gensim/numpy deprecation noise during loading
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

REPO_ROOT = _REPO_ROOT
CANON_ROOT = REPO_ROOT / 'canon-terms'
RESULTS_ROOT = REPO_ROOT / 'results' / 'baseline_C'

# Create output subdirectories
for subdir in ('heatmaps', 'dendrograms', 'barcodes'):
    (RESULTS_ROOT / subdir).mkdir(parents=True, exist_ok=True)

# Domains and languages — single source of truth
DOMAINS = ['color', 'emotion', 'kinship']
LANGS = sorted(SUPPORTED_LANGUAGES)  # deterministic order: ['en', 'es', 'ru']

# Language pairs for cross-lingual analysis (ordered)
LANG_PAIRS = [('en', 'ru'), ('en', 'es'), ('ru', 'es')]

# Language display colors for joint dendrograms and barcodes
LANG_COLORS = {'en': 'steelblue', 'es': 'darkorange', 'ru': 'seagreen'}

print(f'Languages: {LANGS}')
print(f'Domains:   {DOMAINS}')
print(f'Results:   {RESULTS_ROOT}')

Languages: ['en', 'es', 'ru']
Domains:   ['color', 'emotion', 'kinship']
Results:   /home/anna/ph-project-ssa.6/results/baseline_C


## Load MUSE-aligned vectors

In [3]:
# Load MUSE supervised-aligned vectors for all three languages.
# These are text .vec files (no subword fallback) — OOV is expected to be
# higher than CC-300 case.  Each file is ~200 MB in RAM (200K word vocabulary,
# 300 dims).
aligned_vectors = {}
for lang in LANGS:
    print(f'Loading MUSE aligned vectors for lang={lang!r}...')
    kv = load_fasttext(lang, kind='aligned')
    aligned_vectors[lang] = kv
    print(f'  vocab size={len(kv):,}, dim={kv.vector_size}')

print('\nAll MUSE vectors loaded.')

Loading MUSE aligned vectors for lang='en'...


  vocab size=200,000, dim=300
Loading MUSE aligned vectors for lang='es'...


  vocab size=200,000, dim=300
Loading MUSE aligned vectors for lang='ru'...


  vocab size=200,000, dim=300

All MUSE vectors loaded.


## Joint term clouds

In [4]:
def load_canon_terms(lang: str, domain: str) -> list:
    """Read canon term strings from canon-terms/<lang>/<domain>.yaml."""
    path = CANON_ROOT / lang / f'{domain}.yaml'
    with open(path) as f:
        data = yaml.safe_load(f)
    return [entry['term'] for entry in data['terms']]


# Extract term vectors for every (lang, domain) and record OOV stats.
# MUSE aligned vectors have no subword fallback, so misses are expected.

all_terms = {}        # (lang, domain) -> list of all canon terms
matrices = {}         # (lang, domain) -> np.ndarray (n_found, 300)
found_terms_map = {}  # (lang, domain) -> list of found term strings
found_masks = {}      # (lang, domain) -> bool array (n_terms,)
oov_stats = {}        # (lang, domain) -> dict with n_terms, n_found, oov_rate, missing_terms

for lang in LANGS:
    kv = aligned_vectors[lang]
    for domain in DOMAINS:
        terms = load_canon_terms(lang, domain)
        all_terms[(lang, domain)] = terms

        matrix, mask = extract_term_vectors(
            terms, kv, strategy='head', lang=lang, domain=domain
        )
        ft = [t for t, m in zip(terms, mask) if m]
        missing = [t for t, m in zip(terms, mask) if not m]
        n_terms = len(terms)
        n_found = int(mask.sum())
        oov_rate = 1.0 - n_found / n_terms if n_terms else 0.0

        matrices[(lang, domain)] = matrix
        found_terms_map[(lang, domain)] = ft
        found_masks[(lang, domain)] = mask
        oov_stats[(lang, domain)] = {
            'n_terms': n_terms,
            'n_found': n_found,
            'oov_rate': oov_rate,
            'missing_terms': missing,
        }
        print(f'{lang}/{domain}: {n_found}/{n_terms} found, OOV={oov_rate:.1%}'
              + (f'  MISSING: {missing}' if missing else ''))

print('\nTerm vector extraction complete.')

OOV: term='father-in-law', lang='en', domain='kinship' — excluded


OOV: term='mother-in-law', lang='en', domain='kinship' — excluded


OOV: term='son-in-law', lang='en', domain='kinship' — excluded


OOV: term='daughter-in-law', lang='en', domain='kinship' — excluded


OOV: term='brother-in-law', lang='en', domain='kinship' — excluded


...

OOV: term='золовка', lang='ru', domain='kinship' — excluded


OOV: term='свояченица', lang='ru', domain='kinship' — excluded


OOV: term='сноха', lang='ru', domain='kinship' — excluded


OOV: term='свояк', lang='ru', domain='kinship' — excluded


en/color: 11/11 found, OOV=0.0%
en/emotion: 18/18 found, OOV=0.0%
en/kinship: 21/27 found, OOV=22.2%  MISSING: ['father-in-law', 'mother-in-law', 'son-in-law', 'daughter-in-law', 'brother-in-law', 'sister-in-law']
es/color: 11/11 found, OOV=0.0%
es/emotion: 22/22 found, OOV=0.0%
es/kinship: 31/32 found, OOV=3.1%  MISSING: ['concuñada']
ru/color: 12/12 found, OOV=0.0%
ru/emotion: 19/19 found, OOV=0.0%
ru/kinship: 28/34 found, OOV=17.6%  MISSING: ['свёкор', 'деверь', 'золовка', 'свояченица', 'сноха', 'свояк']

Term vector extraction complete.


### OOV Acceptance Gate

Three-tier gate (hard-coded, not configurable):
- **Tier 1** — OOV ≤ 10%: proceed silently
- **Tier 2** — 10% < OOV ≤ 25%: proceed with explicit warning; mark downstream with ⚠; flag for canon-terms followup
- **Tier 3** — OOV > 25%: skip (lang, domain) entirely; record in `oov_skipped.csv`

In [5]:
# OOV acceptance gate — hard-coded thresholds, not configurable.
OOV_TIER2_THRESHOLD = 0.10  # > this → tier-2 (warning)
OOV_TIER3_THRESHOLD = 0.25  # > this → tier-3 (skip)

oov_tiers = {}          # (lang, domain) -> 1, 2, or 3
tier2_pairs = []        # (lang, domain) pairs needing canon-terms followup issue
skipped_pairs = []      # (lang, domain) pairs excluded from all downstream analysis

oov_rows = []           # all rows for oov_rates.csv
skipped_rows = []       # tier-3 rows for oov_skipped.csv

for lang in LANGS:
    for domain in DOMAINS:
        stats = oov_stats[(lang, domain)]
        oov_rate = stats['oov_rate']

        if oov_rate > OOV_TIER3_THRESHOLD:
            tier = 3
            skipped_pairs.append((lang, domain))
            skipped_rows.append({
                'lang': lang, 'domain': domain,
                'n_terms': stats['n_terms'],
                'n_found': stats['n_found'],
                'oov_rate': round(oov_rate, 4),
            })
        elif oov_rate > OOV_TIER2_THRESHOLD:
            tier = 2
            tier2_pairs.append((lang, domain))
        else:
            tier = 1

        oov_tiers[(lang, domain)] = tier
        oov_rows.append({
            'lang': lang, 'domain': domain,
            'n_terms': stats['n_terms'],
            'n_found': stats['n_found'],
            'oov_rate': round(oov_rate, 4),
            'tier': tier,
        })

# Emit tier-2 warnings
for lang, domain in tier2_pairs:
    stats = oov_stats[(lang, domain)]
    print(f'WARNING ⚠  TIER-2 OOV: {lang}/{domain} — '
          f'OOV={stats["oov_rate"]:.1%}  '
          f'({stats["n_found"]}/{stats["n_terms"]} terms found)  '
          f'missing: {stats["missing_terms"]}')
    print(f'  → Downstream results for {lang}/{domain} will be marked ⚠.')
    print(f'  → A discovered-from issue should be filed to revisit canon terms.')

# Emit tier-3 skips
for lang, domain in skipped_pairs:
    stats = oov_stats[(lang, domain)]
    print(f'SKIP ✗   TIER-3 OOV: {lang}/{domain} — '
          f'OOV={stats["oov_rate"]:.1%}  '
          f'({stats["n_found"]}/{stats["n_terms"]} terms found)  '
          f'EXCLUDED from all downstream analyses.')

# Save oov_rates.csv (every (lang, domain) row)
oov_df = pd.DataFrame(oov_rows, columns=['lang', 'domain', 'n_terms', 'n_found', 'oov_rate', 'tier'])
oov_df.to_csv(RESULTS_ROOT / 'oov_rates.csv', index=False)
print(f'\nSaved: oov_rates.csv')

# Save oov_skipped.csv (tier-3 rows only; may be empty)
skip_df = pd.DataFrame(skipped_rows, columns=['lang', 'domain', 'n_terms', 'n_found', 'oov_rate'])
skip_df.to_csv(RESULTS_ROOT / 'oov_skipped.csv', index=False)
print(f'Saved: oov_skipped.csv ({len(skipped_rows)} skipped pairs)')

# Summary
surviving_pairs = [(l, d) for l in LANGS for d in DOMAINS if (l, d) not in skipped_pairs]
print(f'\nOOV gate summary:')
print(f'  Tier 1 (proceed silently): {sum(1 for v in oov_tiers.values() if v == 1)} pairs')
print(f'  Tier 2 (proceed with ⚠):  {len(tier2_pairs)} pairs')
print(f'  Tier 3 (skipped):          {len(skipped_pairs)} pairs')
print(f'  Surviving pairs:           {len(surviving_pairs)}')

# Display OOV table
print()
oov_df

WARNING ⚠  TIER-2 OOV: en/kinship — OOV=22.2%  (21/27 terms found)  missing: ['father-in-law', 'mother-in-law', 'son-in-law', 'daughter-in-law', 'brother-in-law', 'sister-in-law']
  → Downstream results for en/kinship will be marked ⚠.
  → A discovered-from issue should be filed to revisit canon terms.
WARNING ⚠  TIER-2 OOV: ru/kinship — OOV=17.6%  (28/34 terms found)  missing: ['свёкор', 'деверь', 'золовка', 'свояченица', 'сноха', 'свояк']
  → Downstream results for ru/kinship will be marked ⚠.
  → A discovered-from issue should be filed to revisit canon terms.

Saved: oov_rates.csv
Saved: oov_skipped.csv (0 skipped pairs)

OOV gate summary:
  Tier 1 (proceed silently): 7 pairs
  Tier 2 (proceed with ⚠):  2 pairs
  Tier 3 (skipped):          0 pairs
  Surviving pairs:           9



,lang,domain,n_terms,n_found,oov_rate,tier
0,en,color,11,11,0.0000,1
1,en,emotion,18,18,0.0000,1
2,en,kinship,27,21,0.2222,2
3,es,color,11,11,0.0000,1
4,es,emotion,22,22,0.0000,1
5,es,kinship,32,31,0.0312,1
6,ru,color,12,12,0.0000,1
7,ru,emotion,19,19,0.0000,1
8,ru,kinship,34,28,0.1765,2


## Cross-lingual cosine distances

In [6]:
# Cross-lingual cosine distance matrices per domain.
# For each ordered language pair (en,ru), (en,es), (ru,es) and each domain:
#   rows = lang_a found terms, cols = lang_b found terms.
# Uses sklearn.metrics.pairwise.cosine_distances (rectangular; NOT the square
# cosine_distance_matrix from baselines.distances which requires same terms).

cross_dist = {}   # (lang_a, lang_b, domain) -> np.ndarray (n_a, n_b)

# ⚠ marker helper
def oov_marker(lang: str, domain: str) -> str:
    return '⚠' if oov_tiers.get((lang, domain), 1) == 2 else ''


for domain in DOMAINS:
    for lang_a, lang_b in LANG_PAIRS:
        # Skip if either partner is tier-3 (no matrix available)
        if (lang_a, domain) in skipped_pairs or (lang_b, domain) in skipped_pairs:
            print(f'SKIP cross-lingual heatmap: {lang_a}-{lang_b} / {domain} '
                  f'(one or both langs tier-3 for this domain)')
            continue

        X_a = matrices[(lang_a, domain)]   # (n_a, 300)
        X_b = matrices[(lang_b, domain)]   # (n_b, 300)
        terms_a = found_terms_map[(lang_a, domain)]
        terms_b = found_terms_map[(lang_b, domain)]

        D_cross = cosine_distances(X_a, X_b)   # shape (n_a, n_b)
        cross_dist[(lang_a, lang_b, domain)] = D_cross

        marker_a = oov_marker(lang_a, domain)
        marker_b = oov_marker(lang_b, domain)
        label = (f'{lang_a.upper()}{marker_a} ({len(terms_a)} terms) × '
                 f'{lang_b.upper()}{marker_b} ({len(terms_b)} terms)')
        print(f'{domain} / {label}: D shape={D_cross.shape}, '
              f'range=[{D_cross.min():.4f}, {D_cross.max():.4f}]')

        # Plot heatmap
        fig, ax = plt.subplots(figsize=(max(5, len(terms_b) * 0.45),
                                        max(4, len(terms_a) * 0.40)))
        im = ax.imshow(D_cross, vmin=0, vmax=1, aspect='auto', cmap='viridis')
        ax.set_xticks(range(len(terms_b)))
        ax.set_xticklabels(terms_b, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(len(terms_a)))
        ax.set_yticklabels(terms_a, fontsize=7)
        ax.set_xlabel(f'{lang_b.upper()} canon terms', fontsize=9)
        ax.set_ylabel(f'{lang_a.upper()} canon terms', fontsize=9)
        ax.set_title(
            f'Cross-lingual cosine distance — {domain.capitalize()}\n'
            f'{lang_a.upper()}{marker_a} rows × {lang_b.upper()}{marker_b} cols '
            f'(MUSE aligned)',
            fontsize=10
        )
        plt.colorbar(im, ax=ax, label='cosine distance')
        fig.tight_layout()

        out_path = RESULTS_ROOT / 'heatmaps' / f'{domain}_{lang_a}-{lang_b}.png'
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'  Saved: {out_path.name}')

print('\nCross-lingual heatmaps done.')

color / EN (11 terms) × RU (12 terms): D shape=(11, 12), range=[0.2900, 0.6971]
  Saved: color_en-ru.png
color / EN (11 terms) × ES (11 terms): D shape=(11, 11), range=[0.2027, 0.7233]


  Saved: color_en-es.png
color / RU (12 terms) × ES (11 terms): D shape=(12, 11), range=[0.3015, 0.7052]


  Saved: color_ru-es.png
emotion / EN (18 terms) × RU (19 terms): D shape=(18, 19), range=[0.3347, 0.8732]


  Saved: emotion_en-ru.png
emotion / EN (18 terms) × ES (22 terms): D shape=(18, 22), range=[0.2085, 0.8611]
  Saved: emotion_en-es.png
emotion / RU (19 terms) × ES (22 terms): D shape=(19, 22), range=[0.2994, 0.9376]


  Saved: emotion_ru-es.png
kinship / EN⚠ (21 terms) × RU⚠ (28 terms): D shape=(21, 28), range=[0.1452, 0.6723]


  Saved: kinship_en-ru.png
kinship / EN⚠ (21 terms) × ES (31 terms): D shape=(21, 31), range=[0.0947, 0.7808]


  Saved: kinship_en-es.png
kinship / RU⚠ (28 terms) × ES (31 terms): D shape=(28, 31), range=[0.1870, 0.8946]


  Saved: kinship_ru-es.png

Cross-lingual heatmaps done.


## Joint clustering

For each domain, stack found-term vectors across all non-skipped languages
(EN rows, then RU rows, then ES rows — fixed order from LANGS) into a joint
matrix, compute the square cosine distance matrix, then run agglomerative
clustering and plot a language-colored dendrogram.

In [7]:
# Build joint distance matrices per domain.
# Stacking order: EN found terms, then ES found terms, then RU found terms
# (LANGS = ['en', 'es', 'ru'] — alphabetical order from SUPPORTED_LANGUAGES).
# Row labels: list of (lang, term) tuples for leaf annotation.

joint_matrices = {}     # domain -> np.ndarray (n_a, 300)  raw stacked vectors
joint_distance = {}     # domain -> np.ndarray (n_joint, n_joint) square distance
joint_labels = {}       # domain -> list of (lang, term) tuples
joint_lang_ids = {}     # domain -> list of lang strings (parallel to labels)

for domain in DOMAINS:
    surviving_langs = [l for l in LANGS if (l, domain) not in skipped_pairs]
    if len(surviving_langs) < 2:
        print(f'{domain}: fewer than 2 surviving languages after OOV gate '
              f'({surviving_langs}) — skipping joint analysis for this domain.')
        # Record skip in oov_skipped.csv semantics via a note (data already in oov_skipped.csv)
        continue

    rows_X = []
    row_labels = []
    row_langs = []
    for lang in LANGS:   # fixed order
        if (lang, domain) in skipped_pairs:
            continue
        ft = found_terms_map[(lang, domain)]
        X = matrices[(lang, domain)]
        rows_X.append(X)
        row_labels.extend([(lang, t) for t in ft])
        row_langs.extend([lang] * len(ft))

    X_joint = np.vstack(rows_X)            # (n_total, 300)
    D_joint = cosine_distance_matrix(X_joint)  # (n_total, n_total) square

    joint_matrices[domain] = X_joint
    joint_distance[domain] = D_joint
    joint_labels[domain] = row_labels
    joint_lang_ids[domain] = row_langs
    print(f'{domain}: joint matrix shape={X_joint.shape}, D shape={D_joint.shape}, '
          f'langs={surviving_langs}')

print('\nJoint distance matrices computed.')

color: joint matrix shape=(34, 300), D shape=(34, 34), langs=['en', 'es', 'ru']
emotion: joint matrix shape=(59, 300), D shape=(59, 59), langs=['en', 'es', 'ru']
kinship: joint matrix shape=(80, 300), D shape=(80, 80), langs=['en', 'es', 'ru']

Joint distance matrices computed.


In [8]:
# Joint agglomerative clustering dendrograms — one per domain with ≥2 surviving langs.
# Leaves are colored by language (one color per LANGS entry).

for domain in DOMAINS:
    if domain not in joint_distance:
        continue  # fewer than 2 surviving langs; already noted above

    D = joint_distance[domain]
    labels = joint_labels[domain]
    langs_ids = joint_lang_ids[domain]
    n = len(labels)

    Z = agglomerative_linkage(D, method='average')

    # Build leaf labels as "lang:term" strings
    leaf_labels = [f'{lang}:{term}' for lang, term in labels]

    # Color-by-language mapping for leaves
    leaf_colors = [LANG_COLORS[lang] for lang in langs_ids]

    # scipy.cluster.hierarchy.dendrogram with link_color_func
    # We color leaves by overriding link_color_func — apply colors to
    # individual leaf links by rebuilding leaf_colors in dendrogram order.
    # NB: scipy dendrogram returns ivl (leaf labels in display order) and
    # leaves (original indices in display order).

    fig, ax = plt.subplots(figsize=(max(10, n * 0.55), 6))
    dend = dendrogram(
        Z,
        labels=leaf_labels,
        ax=ax,
        leaf_rotation=60,
        leaf_font_size=7,
        above_threshold_color='gray',
        no_plot=False,
    )

    # Color individual x-tick labels by language
    xlabels = ax.get_xticklabels()
    for lbl in xlabels:
        text = lbl.get_text()
        lang_code = text.split(':')[0] if ':' in text else 'en'
        lbl.set_color(LANG_COLORS.get(lang_code, 'black'))

    # Add language color legend
    from matplotlib.patches import Patch
    surviving_langs_d = sorted(set(langs_ids), key=LANGS.index)
    legend_patches = [Patch(color=LANG_COLORS[l], label=l.upper())
                      for l in surviving_langs_d]
    ax.legend(handles=legend_patches, loc='upper right', fontsize=8,
              title='Language', title_fontsize=8)

    ax.set_title(
        f'Joint dendrogram — {domain.capitalize()} domain\n'
        f'(MUSE aligned, average linkage, cosine distance; '
        f'leaves colored by language)',
        fontsize=10
    )
    ax.set_ylabel('Cosine distance')
    fig.tight_layout()

    out_path = RESULTS_ROOT / 'dendrograms' / f'{domain}.png'
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out_path.name}')

print('\nAll joint dendrograms saved.')

Saved: color.png


Saved: emotion.png


Saved: kinship.png

All joint dendrograms saved.


## Joint persistent homology

Run Vietoris–Rips PH (max_dim=1) on the joint distance matrix for each
surviving domain. Plot H0 + H1 barcodes mirroring the baseline_B style.
Comment on H0 count for color (does Russian cluster separately?).

In [9]:
# Barcode plotting helper — mirrors baseline_B's plot_barcode function exactly.
def plot_barcode(ax, bc_dim, title, color='steelblue'):
    """Draw a barcode diagram on ax.  Each bar is a horizontal line [birth, death]."""
    n = len(bc_dim)
    if n == 0:
        ax.set_title(title, fontsize=9)
        ax.text(0.5, 0.5, 'empty', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color='grey')
        ax.set_yticks([])
        return
    # Sort by persistence (death - birth), longest first
    lengths = bc_dim['death'] - bc_dim['birth']
    order = np.argsort(lengths)[::-1]
    for i, idx in enumerate(order):
        ax.plot([bc_dim['birth'][idx], bc_dim['death'][idx]], [i, i],
                color=color, linewidth=2.0, solid_capstyle='butt')
    ax.set_title(title, fontsize=9)
    ax.set_ylabel('bar index', fontsize=7)
    ax.set_xlabel('filtration value', fontsize=7)
    ax.tick_params(labelsize=7)


# Compute joint barcodes and plot
joint_barcodes = {}

for domain in DOMAINS:
    if domain not in joint_distance:
        continue  # fewer than 2 surviving langs

    D = joint_distance[domain]
    langs_ids = joint_lang_ids[domain]
    surviving_langs_d = sorted(set(langs_ids), key=LANGS.index)

    bc = rips_barcode(D, max_dim=1)
    joint_barcodes[domain] = bc

    n_h0 = len(bc[0])
    n_h1 = len(bc[1])
    print(f'{domain}: joint H0 finite bars={n_h0}, H1 finite bars={n_h1}  '
          f'(joint n_terms={D.shape[0]})')

    # Plot: 1 row × 2 cols (H0 | H1)
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4),
                             constrained_layout=True)
    fig.suptitle(
        f'Joint barcode — {domain.capitalize()} domain\n'
        f'(MUSE aligned, Vietoris–Rips; languages: {", ".join(l.upper() for l in surviving_langs_d)})',
        fontsize=11
    )
    plot_barcode(axes[0], bc[0],
                 title=f'H0 (components), n={n_h0}',
                 color='steelblue')
    plot_barcode(axes[1], bc[1],
                 title=f'H1 (loops), n={n_h1}',
                 color='royalblue')

    out_path = RESULTS_ROOT / 'barcodes' / f'{domain}.png'
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {out_path.name}')

print('\nAll joint barcodes computed and saved.')

color: joint H0 finite bars=33, H1 finite bars=16  (joint n_terms=34)
  Saved: color.png
emotion: joint H0 finite bars=58, H1 finite bars=23  (joint n_terms=59)


  Saved: emotion.png
kinship: joint H0 finite bars=79, H1 finite bars=53  (joint n_terms=80)
  Saved: kinship.png

All joint barcodes computed and saved.


### H0 commentary — Color domain: does Russian cluster separately?

The joint H0 barcode for the **color domain** encodes the component structure
of the combined EN + RU + ES color term cloud under MUSE alignment.  Each
finite H0 bar corresponds to a pair of terms that merge later in the filtration
— a longer bar indicates a larger gap (more distinct cluster).

**What to look for:** If Russian's синий/голубой (dark blue / light blue) are
more distant from English "blue" and Spanish "azul" than color terms are from
each other within a language, they will produce long-persisting H0 bars in the
joint barcode.  A distinctly long bar at the filtration scale of cross-lingual
distances (typically 0.3–0.6 for MUSE) vs. within-language distances (~0.1–0.4
for CC-300) would be consistent with Russian encoding a cross-lingual structural
split in the blue region.

The joint H0 count will be (n_total − 1) finite bars regardless of structure
(that's the nature of H0 — one bar per merged component).  What matters is the
**persistence distribution**: are some H0 bars much longer than others, and do
the longest-persisting ones correspond to the синий/голубой entries?  That
question is better answered by the dendrogram (leaf colors) than by the barcode
scalar count.  See the color dendrogram above — if the Russian blue terms appear
in a separate sub-tree from EN/ES blue, that is the structural signal.

## Prediction #2 descriptive check

**P2:** Mean cross-lingual cosine distance in the EMOTION domain > COLOR domain,
per language pair.  Build a table and bar chart from the cross-lingual distance
matrices computed in step 5 above.

In [10]:
# Prediction #2 descriptive check.
# For each surviving language pair × domain, compute mean of cross-lingual D matrix.
# Mark with ⚠ if either side of the pair was tier-2 for that domain.

pred2_rows = []

for domain in DOMAINS:
    for lang_a, lang_b in LANG_PAIRS:
        if (lang_a, domain) in skipped_pairs or (lang_b, domain) in skipped_pairs:
            continue  # tier-3: no cross-lingual matrix available
        if (lang_a, lang_b, domain) not in cross_dist:
            continue

        D_cross = cross_dist[(lang_a, lang_b, domain)]
        mean_dist = float(D_cross.mean())

        # Marker: ⚠ if either lang is tier-2 for this domain
        marker = ''
        if oov_tiers.get((lang_a, domain), 1) == 2 or oov_tiers.get((lang_b, domain), 1) == 2:
            marker = '⚠'

        pair_str = f'{lang_a.upper()}-{lang_b.upper()}'
        pred2_rows.append({
            'lang_pair': pair_str,
            'domain': domain,
            'mean_distance': round(mean_dist, 4),
            'marker': marker,
        })

pred2_df = pd.DataFrame(pred2_rows, columns=['lang_pair', 'domain', 'mean_distance', 'marker'])
pred2_df.to_csv(RESULTS_ROOT / 'pred2.csv', index=False)
print(f'Saved: pred2.csv')
pred2_df

Saved: pred2.csv


,lang_pair,domain,mean_distance,marker
0,EN-RU,color,0.4795,
1,EN-ES,color,0.4929,
2,RU-ES,color,0.4837,
3,EN-RU,emotion,0.5938,
4,EN-ES,emotion,0.5604,
5,RU-ES,emotion,0.6108,
6,EN-RU,kinship,0.4049,⚠
7,EN-ES,kinship,0.3770,⚠
8,RU-ES,kinship,0.4659,⚠


In [11]:
# Grouped bar chart: one group per language pair, three bars per group (domains).
# Domains in a fixed color mapping for the bar chart.
DOMAIN_COLORS = {'color': '#4878cf', 'emotion': '#d65f5f', 'kinship': '#6acc65'}

pairs_in_data = pred2_df['lang_pair'].unique().tolist()
# Maintain fixed pair order
pair_order = [f'{a.upper()}-{b.upper()}' for a, b in LANG_PAIRS
              if f'{a.upper()}-{b.upper()}' in pairs_in_data]

x = np.arange(len(pair_order))
width = 0.25
n_domains = len(DOMAINS)
offsets = np.linspace(-(n_domains - 1) * width / 2,
                       (n_domains - 1) * width / 2,
                       n_domains)

fig, ax = plt.subplots(figsize=(9, 5))

for i, domain in enumerate(DOMAINS):
    sub = pred2_df[pred2_df['domain'] == domain]
    means = []
    for pair in pair_order:
        row = sub[sub['lang_pair'] == pair]
        means.append(row['mean_distance'].values[0] if len(row) else np.nan)

    bars = ax.bar(x + offsets[i], means, width,
                  label=domain.capitalize(),
                  color=DOMAIN_COLORS[domain],
                  alpha=0.85)

    # Add ⚠ text above bars where marker is set
    for j, pair in enumerate(pair_order):
        row = sub[sub['lang_pair'] == pair]
        if len(row) and row['marker'].values[0] == '⚠':
            bar_height = means[j] if not np.isnan(means[j]) else 0
            ax.text(x[j] + offsets[i], bar_height + 0.005, '⚠',
                    ha='center', va='bottom', fontsize=8, color='darkorange')

ax.set_xticks(x)
ax.set_xticklabels(pair_order, fontsize=10)
ax.set_ylabel('Mean cross-lingual cosine distance', fontsize=10)
ax.set_title(
    'Prediction #2 descriptive check\n'
    'Mean cross-lingual cosine distance by language pair and domain\n'
    '(MUSE aligned; ⚠ = tier-2 OOV on one or both sides)',
    fontsize=10
)
ax.legend(fontsize=9)
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
fig.tight_layout()

pred2_bar_path = RESULTS_ROOT / 'pred2_bar.png'
fig.savefig(pred2_bar_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: pred2_bar.png')

Saved: pred2_bar.png


## Discussion

### OOV outcomes per language

MUSE aligned vectors have no subword fallback, so OOV rates are higher than in
the CC-300 case.  Multi-word terms (e.g., kinship compounds like "maternal uncle",
"двоюродный брат", "tío materno") are handled by the head-word strategy, so
they depend on the MUSE vocabulary covering the head word.  Color and basic
emotion terms are typically single-word and should fare better.

Tier-2 pairs (OOV 10–25%) are still usable but downstream distances are noisy;
results are marked ⚠ and flagged for canon-terms review.  Tier-3 pairs (OOV > 25%)
are excluded — their sparse coverage would make the distance matrix unrepresentative
of the semantic domain.

### Prediction #2 direction

**P2:** Emotion cross-lingual divergence > color cross-lingual divergence.

The bar chart above (pred2_bar.png) gives the qualitative picture.  If the
emotion bar is taller than the color bar for most or all language pairs, P2
has descriptive support in the MUSE-aligned baseline.  Formal testing happens
in Phase 4 (ph-project-bt5).

Note: color terms are relatively universal (Berlin & Kay), so we expect lower
cross-lingual distances for color than for emotion (where culturally-specific
concepts like тоска, duende, ilusión appear).  Kinship terms, which also carry
strong cross-linguistic divergence due to typological variation (Eskimo-type
with gender-marked vs. gender-neutral distinctions), may rival emotion divergence.

### Limitations

This is a static embedding baseline using MUSE supervised alignment.  MUSE
alignment is trained on bilingual dictionaries, so it tends to align translation
equivalents closely — this may *underestimate* true semantic divergence for
culturally-specific terms that have imprecise dictionary translations.  The main
experiment (Phase 3, mBERT attention graphs) will capture contextual structure
not available in static embeddings.

In [12]:
# Artifact verification — confirm all expected outputs exist.
import itertools

# Expected heatmaps: one per surviving lang pair × domain
expected_heatmaps = [
    RESULTS_ROOT / 'heatmaps' / f'{domain}_{la}-{lb}.png'
    for domain in DOMAINS
    for la, lb in LANG_PAIRS
    if (la, domain) not in skipped_pairs and (lb, domain) not in skipped_pairs
    and (la, lb, domain) in cross_dist
]

# Expected dendrograms and barcodes: one per domain with ≥2 surviving langs
expected_dendrograms = [
    RESULTS_ROOT / 'dendrograms' / f'{domain}.png'
    for domain in joint_distance
]
expected_barcodes = [
    RESULTS_ROOT / 'barcodes' / f'{domain}.png'
    for domain in joint_barcodes
]

expected_csvs = [
    RESULTS_ROOT / 'oov_rates.csv',
    RESULTS_ROOT / 'oov_skipped.csv',
    RESULTS_ROOT / 'pred2.csv',
    RESULTS_ROOT / 'pred2_bar.png',
]

all_expected = expected_heatmaps + expected_dendrograms + expected_barcodes + expected_csvs

missing = [str(p) for p in all_expected if not p.exists()]

if missing:
    print(f'MISSING ARTIFACTS ({len(missing)}):')
    for m in missing:
        print(f'  {m}')
else:
    print(f'All artifacts present:')
    print(f'  {len(expected_heatmaps)} cross-lingual heatmaps')
    print(f'  {len(expected_dendrograms)} joint dendrograms')
    print(f'  {len(expected_barcodes)} joint barcodes')
    print(f'  {len(expected_csvs)} CSVs / summary files')

assert not missing, f'Missing artifacts: {missing}'

All artifacts present:
  9 cross-lingual heatmaps
  3 joint dendrograms
  3 joint barcodes
  4 CSVs / summary files
